# 02 — PCP: collect → train Q-corrector → 3-way eval

Same orchestration pattern as notebook 01. Collection is just a rollout with the
`save_pcp_features` sink; the 3-way eval is three ordinary method rows (`vanilla` /
`pnp_only` / `pcp`). No qc_rollouts / qc_eval tables — everything lands in `rollouts`.

## 1. Secrets + install

In [ ]:
import os
from google.colab import userdata
for k in ('SUPABASE_URL', 'SUPABASE_SERVICE_KEY', 'HF_TOKEN', 'WANDB_API_KEY'):
    os.environ[k] = userdata.get(k)
GH_PAT = userdata.get('GH_PAT')
![ -d pnp-vla ] || git clone -q https://$GH_PAT@github.com/ArjunS07/pnp-vla.git
!cd pnp-vla && git pull -q && pip install -q -e '.[sim]'

## 2. Environment + model + store + PRO episodes

In [ ]:
from pnp.env_setup import setup_environment
setup_environment()

In [ ]:
from pnp import libero_env, libero_pro, models, RolloutConfig, TrainConfig, Method
from pnp.store import SupabaseStore
from pnp.rollout import run_episode, iter_task_envs

libero_env.init_libero_benchmark()
policy, preprocess, postprocess = models.load_pi05()
device = models.default_device()
store = SupabaseStore()
EXPERIMENT = 'pcp-v1'
train_cfg = TrainConfig()
PNP_K = 3
COLLECT_STEPS = tuple(range(10))   # probe broadly at collect time; training filters to deploy steps

# libero_pro.apply_env_patches(); libero_pro.patch_torch_load()
# bd = libero_pro.reload_benchmark()
# pro_eps = libero_pro.build_libero_pro_episodes(bd, episode_idxs=range(10))
pro_eps = []   # <- fill from libero_pro once assets are set up

## 3. Collect labeled (z_hat, obs_enc) chunks → rollouts (save_pcp_features sink)

In [ ]:
# Collection = a vanilla rollout (no action) with the pcp-features sink on. The probe
# supplies z_hat + obs_enc at COLLECT_STEPS; the label is the rollout's success.
COLLECT = RolloutConfig(pnp_steps=COLLECT_STEPS, pnp_k=PNP_K,
                        save_pcp_features=True, save_trajectory=False)
store.start_run(driver='pcp_collect', benchmark='libero_pro', experiment=EXPERIMENT)
done = store.existing_keys(EXPERIMENT)
for env, task_eps in iter_task_envs(pro_eps):
    for ep, name, cfg, rid in store.iter_todo(EXPERIMENT, task_eps, [(Method.COLLECT, COLLECT)], done=done):
        res = run_episode(env, ep, policy, preprocess, device, cfg)
        store.log_result(rid, ep, name, cfg, res)
store.finish_run()

## 4. Train + calibrate the Q-corrector (wandb) → q_correctors

Training data is read straight back from `rollouts` (every row with a `pcp_chunks_path`).

In [ ]:
import wandb
from pnp.pcp import load_qc_samples, train_q_corrector, ckpt_bytes, new_q_ckpt_id

qc_rows = store.load_qc_rows(experiment=EXPERIMENT)          # rollouts WHERE pcp_chunks_path NOT NULL
samples = load_qc_samples(qc_rows, train_cfg)
run = wandb.init(project='pnp-qcorrector', name=EXPERIMENT, config={'experiment': EXPERIMENT})
q_model, q_scaler, meta, split_ids = train_q_corrector(
    samples, device, cfg=train_cfg, wandb_run=run, experiment=EXPERIMENT)
q_ckpt_id = new_q_ckpt_id()
store.register_q_corrector(q_ckpt_id, ckpt_bytes(q_model, q_scaler, meta), meta, split_ids=split_ids)
run.finish()
print('q_ckpt_id =', q_ckpt_id, ' val_auc =', meta['val_auc'])

## 5. 3-way eval (vanilla / pnp_only / pcp) → three rollouts method rows

The three arms are just `RolloutConfig`s: vanilla, a correction with λ=0 (P&P refine,
no gradient), and λ>0 (full PCP). The corrector is attached via the `q_model`/`q_scaler`
runtime handles. All three are paired on the same `(suite, task, episode, init_state)` noise,
so SR is comparable.

In [ ]:
from pnp.pcp import QCorrector, TemperatureScaler

ckpt, _ = store.load_q_corrector(q_ckpt_id)
q_model = QCorrector(ckpt['action_dim'], ckpt['obs_dim']).to(device)
q_model.load_state_dict(ckpt['model']); q_model.eval()
q_scaler = TemperatureScaler().to(device); q_scaler.load_state_dict(ckpt['scaler'])

CORR_STEPS = tuple(train_cfg.correction_steps)
def correct(lam):
    return RolloutConfig(pnp_steps=CORR_STEPS, pnp_k=PNP_K, correction_lambda=lam, q_gate=0.5,
                         q_ckpt_id=q_ckpt_id, q_model=q_model, q_scaler=q_scaler)

PASSES = {
    Method.VANILLA:  RolloutConfig(),
    Method.PNP_ONLY: correct(0.0),   # P&P refine at CORR_STEPS, no gradient
    Method.PCP:      correct(3.0),   # full PCP
}
store.start_run(driver='pcp_eval', benchmark='libero_pro', experiment=EXPERIMENT)
done = store.existing_keys(EXPERIMENT)
for env, task_eps in iter_task_envs(pro_eps):
    for ep, name, cfg, rid in store.iter_todo(EXPERIMENT, task_eps, PASSES, done=done):
        res = run_episode(env, ep, policy, preprocess, device, cfg)
        store.log_result(rid, ep, name, cfg, res)
store.finish_run()